In [0]:
%run ./00_config

In [0]:
%run ./00_functions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Read silver tables from workspace.bakehouse_training
silver_transactions_df = spark.read.table(silver_transactions_fact)
silver_franchises_dim = spark.read.table(silver_franchises_dim)

In [0]:
# Join the sales and franchises tables
transactions_franchises_df = enrich_with_franchises(
    silver_transactions_df, 
    silver_franchises_dim,
    cols_franchises = ['franchiseID', 'franchiseName', 'city', 'district', 'country', 'size'])

In [0]:
# Create the gold table 
daily_sales_franchises_metrics = (
    transactions_franchises_df
    .groupBy('franchiseID', 'franchiseName', 'transaction_date')
    .agg(F.sum('totalPrice').alias('sum_revenue'),
         F.round(F.avg('totalPrice'), 2).alias('avg_revenue'),
         F.countDistinct('customerID').alias('amount_of_customers')
         )
)
daily_sales_franchises_metrics = (
    daily_sales_franchises_metrics.withColumn(
        'rank_sum_revenue',
        F.dense_rank().over(Window.partitionBy('transaction_date').orderBy(F.desc('sum_revenue')))
    )
)

In [0]:
daily_sales_franchises_metrics.display()

In [0]:
# Check the gold table : key uniqueness and total revenue against Silver 
check_gold_key_uniqueness(daily_sales_franchises_metrics, ['franchiseID', 'transaction_date'])
check_gold_silver_sum_revenue(silver_transactions_df, 'totalPrice', daily_sales_franchises_metrics,'sum_revenue')

In [0]:
# Create gold table from SQL 
gold_sql = spark.sql(
    """
    WITH daily_sales_frandises AS(
        SELECT f.franchiseID, f.franchiseName, t.transaction_date, 
           SUM(t.totalPrice) AS sum_revenue, 
           ROUND(AVG(t.totalPrice), 2) AS avg_revenue, 
           COUNT(DISTINCT t.customerID) AS amount_of_customers
        FROM workspace.bakehouse_training.silver_transactions_fact t
        LEFT JOIN workspace.bakehouse_training.silver_franchises_dim f USING(franchiseID)
        GROUP BY f.franchiseID, f.franchiseName, t.transaction_date
    )
    SELECT *, DENSE_RANK() OVER(PARTITION BY transaction_date ORDER BY sum_revenue DESC) AS rank_sum_revenue
    FROM daily_sales_frandises
    """
)
gold_pyspark = daily_sales_franchises_metrics

In [0]:
# Check the output from PySpark and SQL are the same with exceptAll
r = check_except_all_double_directions(gold_sql, gold_pyspark)
print('All rows are same in both PySpark and SQL output')
write_gold(daily_sales_franchises_metrics, gold_transactions_metrics)